# Figure 4 — pages (Supplementary Figure 4Sb)
Assembles the panels drawn by `muscle_figures.ipynb` into the 4Sb page, cropped tight → page SVG in
`../../illustrator-figures/` and SVG/PDF/PNG in `figures/`. Figure 4 and 4Sa are assembled in Illustrator
(`../../illustrator-figures/Figure4.ai`, `Figure4Sa.ai`); the 4S NIA page is made by `nia_replication.ipynb`.

In [1]:
import copy
import io
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
from PIL import Image

FIG = "figures"
PAGES = "../../illustrator-figures"
SVG_NS = "http://www.w3.org/2000/svg"
for prefix, uri in [("", SVG_NS), ("xlink", "http://www.w3.org/1999/xlink"), ("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"),
                    ("cc", "http://creativecommons.org/ns#"), ("dc", "http://purl.org/dc/elements/1.1/")]:
    ET.register_namespace(prefix, uri)
PAGE_W, PAGE_H = 595.276, 841.89                       # A4 (pt) layout frame; cropped afterwards
CONTENT_X, CONTENT_W, GAP_X, GAP_Y, PAD = 28, 543, 18, 18, 4


def load(name):
    """Root element of a panel SVG in figures/."""
    return ET.parse(f"{FIG}/{name}.svg").getroot()


def new_page():
    """Blank A4 page with a white background."""
    page = ET.Element(f"{{{SVG_NS}}}svg", width=f"{PAGE_W}pt", height=f"{PAGE_H}pt", viewBox=f"0 0 {PAGE_W} {PAGE_H}")
    ET.SubElement(page, f"{{{SVG_NS}}}rect", id="background", x="0", y="0", width=f"{PAGE_W}", height=f"{PAGE_H}",
                  fill="#ffffff")
    return page


def nest(page, root, x, y, scale):
    """Place a panel SVG on the page at (x, y), scaled. Returns its bottom."""
    vb = [float(v) for v in root.get("viewBox").split()]
    child = ET.SubElement(page, f"{{{SVG_NS}}}svg", x=f"{x}", y=f"{y}", width=f"{vb[2] * scale}", height=f"{vb[3] * scale}",
                          viewBox=" ".join(f"{v}" for v in vb), preserveAspectRatio="xMinYMin meet", overflow="hidden")
    for el in root:
        child.append(copy.deepcopy(el))
    return y + vb[3] * scale


def letter(page, text, x, y):
    """Panel letter (Arial 12, lowercase + period)."""
    t = ET.SubElement(page, f"{{{SVG_NS}}}text", x=f"{x}", y=f"{y}",
                      style="font-family: Arial, ArialMT, sans-serif; font-size: 12px; fill: #000000")
    t.text = f"{text}."


def pair_row(page, letters, names, y):
    """Two panels side by side at one shared scale, top-aligned, spanning the content width. Returns the row bottom."""
    roots = [load(n) for n in names]
    widths = [float(r.get("viewBox").split()[2]) for r in roots]
    scale = (CONTENT_W - GAP_X) / sum(widths)
    x, bottom = CONTENT_X, y
    for lt, root, w in zip(letters, roots, widths):
        bottom = max(bottom, nest(page, root, x, y, scale))
        letter(page, lt, x - (11 if x == CONTENT_X else 4), y + 8)
        x += w * scale + GAP_X
    return bottom


def write(page, name):
    """Crop to the drawn content (+ PAD); save the SVG to illustrator-figures and SVG/PDF/PNG to figures/."""
    png = cairosvg.svg2png(bytestring=ET.tostring(page), scale=4)
    ink = np.asarray(Image.open(io.BytesIO(png)).convert("L")) < 250
    px = ink.shape[1] / PAGE_W
    rows = np.where(ink.any(axis=1))[0]
    cols = np.where(ink.any(axis=0))[0]
    x0, x1 = max(cols[0] / px - PAD, 0), min((cols[-1] + 1) / px + PAD, PAGE_W)
    y0, y1 = max(rows[0] / px - PAD, 0), min((rows[-1] + 1) / px + PAD, PAGE_H)
    page.set("viewBox", f"{x0:.2f} {y0:.2f} {x1 - x0:.2f} {y1 - y0:.2f}")
    page.set("width", f"{x1 - x0:.2f}pt")
    page.set("height", f"{y1 - y0:.2f}pt")
    page.find(f"{{{SVG_NS}}}rect[@id='background']").attrib.update(x=f"{x0:.2f}", y=f"{y0:.2f}", width=f"{x1 - x0:.2f}",
                                                                    height=f"{y1 - y0:.2f}")
    for path in (f"{PAGES}/{name}.svg", f"{FIG}/{name}.svg"):
        ET.ElementTree(page).write(path, xml_declaration=True, encoding="utf-8")
    cairosvg.svg2pdf(url=f"{FIG}/{name}.svg", write_to=f"{FIG}/{name}.pdf")
    cairosvg.svg2png(url=f"{FIG}/{name}.svg", write_to=f"{FIG}/{name}.png", scale=2)
    print(f"{name}: {x1 - x0:.0f} x {y1 - y0:.0f} pt")

## Supplementary Figure 4Sb
Row 1: a | b at one shared scale, top-aligned; row 2: c across the full width.

In [2]:
page = new_page()
bottom = pair_row(page, "ab", ["fig4Sb_a_locus_matched_young_vs_geriatric", "fig4Sb_b_locus_matched_aged_vs_geriatric"], 17)
cores = load("fig4Sb_c_ifna_complement_cores")
y = bottom + GAP_Y
letter(page, "c", CONTENT_X - 11, y + 8)
nest(page, cores, CONTENT_X, y, CONTENT_W / float(cores.get("viewBox").split()[2]))
write(page, "Figure4Sb")

Figure4Sb: 559 x 515 pt
